In [37]:
import os
from dotenv import load_dotenv

from langchain_openai import ChatOpenAI
from langchain_core.tools import Tool
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.prompts import PromptTemplate 
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel
from langchain.tools import Tool
from langchain.memory import ConversationBufferMemory
from langchain.agents import initialize_agent, AgentType


In [38]:
llm = ChatOpenAI(model="gpt-3.5-turbo",temperature=0.3)
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)


In [39]:
def get_weather(city: str) -> str:
    weather_data = {
        "paris": "18°C, cloudy",
        "london": "15°C, rainy",
        "nairobi": "24°C, sunny",
        "calgary": "20°C, partly cloudy",
        "tokyo": "27°C, humid"
    }

    return weather_data.get(
        city.lower(),
        f"No weather data available for {city}"
    )
weather_tool = Tool(
    name="WeatherTool",
    func=get_weather,
    description="Returns weather information for a city."
)

In [40]:
def get_currency_rate(query: str) -> str:
    rates = {
        "usd_to_eur": "0.92",
        "usd_to_cad": "1.37",
        "usd_to_gbp": "0.79",
        "usd_to_kes": "129.50"
    }

    return rates.get(
        query.lower(),
        "Currency conversion not available."
    )

currency_tool = Tool(
    name="CurrencyTool",
    func=get_currency_rate,
    description="Returns currency conversion rates."
)

In [41]:

tools = [weather_tool, currency_tool]

In [42]:
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION,
    memory=memory,
    verbose=False,
    handle_parsing_errors="Check your output format. Must use Action/Action Input."

)

In [43]:
recommendation_prompt = PromptTemplate.from_template("""You are a travel expert. Give 3 recommendations for visiting {city}.""")
recommendation_chain = (recommendation_prompt | llm | StrOutputParser())

budget_prompt = PromptTemplate.from_template("""You are a travel budget expert. Give budget-saving tips for visiting {city}.""")
budget_chain = ( budget_prompt | llm | StrOutputParser())

parallel_chain = RunnableParallel({"recommendations": recommendation_chain, "budget": budget_chain})


In [44]:
print("Smart Travel Assistant Ready (type 'exit' to stop)\n")
known_cities = ["calgary", "london", "paris", "tokyo", "nairobi", "new york", "sydney", "rome", "barcelona", "dubai", "singapore", "hong kong", "bangkok", "los angeles", "chicago", "miami", "boston", "seattle", "vancouver", "toronto"]

while True:
    user_input = input("You: ")
    if user_input.lower() == "exit":
        break
    # Agent response (with tools + memory)
    response = agent.invoke({"input": user_input})
    print("\nAssistant:", response["output"])
    # Detect city for parallel chain
    city = None
    for c in known_cities:
        if c in user_input.lower():
            city = c.title()
            break
    # Run parallel chains
    if city:
        result = parallel_chain.invoke({"city": city})
        print("\n--- Recommendations ---")
        print(result["recommendations"])
        print("\n--- Budget Advice ---")
        print(result["budget"])

Smart Travel Assistant Ready (type 'exit' to stop)


Assistant: The weather in London is currently 15°C and rainy, so it may not be the best time to travel there.

--- Recommendations ---
1. Explore the iconic landmarks: Make sure to visit famous landmarks such as Big Ben, Buckingham Palace, the Tower of London, and the London Eye. These attractions offer a glimpse into the city's rich history and culture.

2. Experience the diverse food scene: London is a melting pot of different cultures, which is reflected in its vibrant food scene. Be sure to try traditional British dishes like fish and chips, as well as explore the city's many international cuisines in areas like Chinatown and Brick Lane.

3. Take advantage of the city's world-class museums and galleries: London is home to some of the best museums and galleries in the world, many of which offer free admission. Don't miss the British Museum, the Tate Modern, and the National Gallery for a dose of art and culture during your visit.


In [45]:
print(memory.load_memory_variables({}))

{'chat_history': [HumanMessage(content='is london great to travel now'), AIMessage(content='The weather in London is currently 15°C and rainy, so it may not be the best time to travel there.'), HumanMessage(content='what about new york'), AIMessage(content='The weather in London is 15°C and rainy.'), HumanMessage(content='can i travel to Nairobi'), AIMessage(content='Yes, you can travel to Nairobi.'), HumanMessage(content='what did i ask you previously'), AIMessage(content='N/A')]}
